# Working with Dates in PySpark

Dates almost never arrive in a clean, typed form. They come as separate year/month/day columns, as strings in odd formats, or with invalid values mixed in. This notebook walks through the core date workflow in Spark:

| # | Step | Key functions | Why it matters |
|---|------|---------------|----------------|
| 1 | Build a test DataFrame | `concat_ws` | Creates a mix of valid **and** invalid dates to see how Spark reacts |
| 2 | Convert string → date | `try_to_date` (via `expr`) | Turns text into a real `DateType` so date math works, and safely nulls out bad values |
| 3 | Date arithmetic | `date_add`, `date_sub`, `add_months` | Shift dates forward/backward (due dates, windows, offsets) |
| 4 | Today & differences | `current_date`, `date_diff`, `date - date` | Measure age, duration, time since an event |
| 5 | Formatting | `date_format` | Turn a date back into a human-readable string for reports/exports |

**Reference:** [PySpark date & timestamp functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#date-and-timestamp-functions) · [Datetime pattern letters](https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html)

> **Environment note:** This notebook was written for **Databricks**, where the `spark` session and the `.display()` method are provided automatically. In plain PySpark, create `spark` yourself (`SparkSession.builder.getOrCreate()`) and use `.show()` instead of `.display()`.

## 1. Requirement: build a DataFrame of candidate dates

We start with **year, month and day as separate integer columns** (common in raw/legacy data) and glue them into one string like `2022-5-18`.

The rows were picked on purpose. Each one tests a different edge case:

| Input (Y, M, D) | What it tests |
|-----------------|---------------|
| `2022, 5, 18`   | A normal, valid date (baseline) |
| `9999, 12, 31`  | The **maximum** date Spark officially supports |
| `-9999, 1, 1`   | A negative (BC) year |
| `10000, 1, 1`   | A year **above** the supported range |
| `-10000, 1, 1`  | A year **below** the supported range |
| `193, 5, 25`    | A 3-digit year (should be year 193 AD, not 1993) |
| `99, 5, 25`     | A 2-digit year (should be year 99 AD, not 1999/2099) |
| `1000, 2, 29`   | Feb 29 in year 1000, which is **not** a leap year in the Gregorian calendar (see section 2) |

In [ ]:
# concat_ws = "concatenate with separator".
# WHY: it joins several columns into one string, putting the separator ("-") between them.
#      It also casts the integer columns to strings automatically and skips NULLs,
#      which makes it safer than plain concat() (concat returns NULL if ANY input is NULL).
from pyspark.sql.functions import concat_ws

# Test data: tuples of (year, month, day). Includes valid AND deliberately invalid dates.
data_list = [(2022, 5, 18) , (9999, 12, 31), (-9999, 1, 1), (10000, 1, 1), 
             (-10000, 1, 1), (193, 5, 25),   (99, 5, 25),   (1000, 2, 29)]

df = (spark.createDataFrame(data_list)          # Spark infers column names _1, _2, _3
     .toDF("Y", "M", "D")                      # toDF(...) renames ALL columns in order -> Y, M, D
     .withColumn("date_str",                   # add a new column...
                 concat_ws("-", "Y", "M", "D"))) # ...e.g. 2022, 5, 18 -> "2022-5-18" (string, NOT a date yet)

# At this point date_str is just text: you can't add days to it or compare it as a date.
df.display()

## 2. Convert strings to dates

**Goal:** turn `date_str` (a `StringType`) into a real `DateType` column. Only a real date type supports date arithmetic, correct sorting and comparisons, and date functions.

### How Spark decides if a date is valid
1. **Proleptic Gregorian calendar.** Since Spark 3.0, Spark applies the Gregorian calendar rules to *all* dates, including those before 1582 when the Gregorian calendar was actually introduced. (Spark 2.x used a hybrid Julian + Gregorian calendar.)
   - Example: `1000-02-29`. Under the **Julian** calendar year 1000 is a leap year, so this date would exist. Under the **Gregorian** rule (a century year must be divisible by 400 to be a leap year) it is **not**, so Spark treats it as **invalid**.
2. **Negative years are BC, positive years are AD** in the Gregorian calendar.
3. **Invalid or out-of-range dates** either **throw an exception** or become **`NULL`**, depending on which function you use:

| Function | On an invalid date | When to use |
|----------|-------------------|-------------|
| `to_date(col, fmt)` | Throws an error when ANSI mode is on (the default in Spark 4 and newer Databricks runtimes). Returns NULL in legacy/non-ANSI mode | When bad data should **fail the job** (strict pipelines) |
| `try_to_date(col, fmt)` | Always returns `NULL` | When you want to **keep the row** and flag/clean bad values later |

### The format pattern `'y-M-d'`
- `y` = year, `M` = month (**capital M**; lowercase `m` means *minutes*!), `d` = day of month.
- A **single letter** accepts a variable number of digits, so `5` and `05` both parse. With `'yyyy-MM-dd'`, the input would have to be zero-padded exactly.

### Why `expr(...)`?
`try_to_date` was added as a **SQL** function (Spark 3.5) before a matching Python wrapper existed in `pyspark.sql.functions`. `expr()` lets you call **any Spark SQL function** from PySpark by writing it as a SQL string. This is a handy workaround whenever a SQL function has no Python equivalent in your Spark version.

In [ ]:
# expr() parses a SQL expression string into a Column.
# WHY: lets us use the SQL function try_to_date even if there is no Python wrapper for it.
from pyspark.sql.functions import expr

df = (
    # try_to_date(string, pattern):
    #   - we can't use cast -> because there are different formats, we only use it for default spark format
    #   - expr will handle the try_to_date so we don't need to import it 
    #   - valid date string   -> DateType value (e.g. "2022-5-18" -> 2022-05-18)
    #   - invalid / unparsable -> NULL (instead of crashing the whole job like to_date would in ANSI mode)
    # 'y-M-d' = year-month-day, single letters so non-padded values like "5" are accepted.
    df.withColumn("valid_date", expr("try_to_date(date_str, 'y-M-d')"))
        # WHY drop: Y, M, D were only needed to build date_str. Removing them keeps the output focused.
        .drop("Y", "M", "D")
)

# Check the output: rows whose valid_date is NULL are the ones Spark rejected
# (e.g. 1000-2-29, which is not a real Gregorian date).
df.display()

## 3. Add and subtract days and months

**Goal:** shift dates forward or backward, e.g. *invoice due date = issue date + 30 days* or *the last 6 months of data*.

| Function | What it does | Notes |
|----------|--------------|-------|
| `date_add(date, n)` | date **+ n days** | `n` can be negative |
| `date_sub(date, n)` | date **− n days** | Same as `date_add(date, -n)`. Use whichever reads more clearly |
| `add_months(date, n)` | date **+ n months** | No `sub_months` exists: pass a **negative n** to go back |

**Why not just add 30 days for a month?** Months have 28–31 days. `add_months` is calendar-aware: if the result day doesn't exist, it **clamps to the last day of the month**. For example, `add_months('2024-01-31', 1)` returns `2024-02-29`, not an error and not March 2.

**NULL in, NULL out:** rows where `valid_date` is NULL simply produce NULL. No errors.

In [ ]:
from pyspark.sql.functions import date_add, date_sub, add_months, date_diff

df = (
    # withColumns({name: expression, ...}) adds/replaces SEVERAL columns in one call.
    # WHY: cleaner and more efficient than chaining 4 separate withColumn() calls
    #      (each withColumn adds another projection to the query plan).
    df.withColumns({
        "add_5_days": date_add("valid_date", 5),        # +5 calendar days
        "sub_5_days": date_sub("valid_date", 5),        # -5 calendar days
        "add_5_months": add_months("valid_date", 5),    # +5 months (clamps to month end if needed)
        "sub_5_months": add_months("valid_date", -5)    # -5 months: there is no sub_months, so use a negative number
    })
)

df.display()

## 4. Current date, date difference, and intervals

**Goal:** measure the time *between* two dates, e.g. customer age, days since last order, or SLA breaches.

| Expression | Returns | Use it when |
|-----------|---------|-------------|
| `current_date()` | Today's date (`DateType`), in the session time zone | Comparing data against "today" |
| `date_diff(end, start)` | **Integer** number of days, `end − start` | You need a plain number (filtering, aggregating, e.g. `> 30`) |
| `col(date1) - col(date2)` | An **interval** (`INTERVAL DAY`) | You want a typed duration you can add back to dates/timestamps |

**Watch the argument order:** `date_diff(end, start)`. The **later** date goes **first**. If you swap them, the result is negative.

> `current_date()` is evaluated **once per query**, so every row gets the same value even on a large cluster. This keeps results consistent.

> `date_diff` is the newer name (Spark 3.5+) for the older `datediff`. Both behave the same.

In [ ]:
from pyspark.sql.functions import current_date, date_diff, col
df = (
    df.withColumns({
        # Today's date, the same value for every row in this query.
        "current_date": current_date(),

        # Number of days between the two dates as an INTEGER (end first, start second).
        # add_5_months - valid_date is roughly 150-153 days, depending on the months crossed.
        "delta_date_days": date_diff("add_5_months", "valid_date"),

        # Subtracting two DATE columns gives an INTERVAL (e.g. "INTERVAL '1234' DAY") instead of a number.
        # WHY: an interval is a proper duration type, so it can be added to dates/timestamps later.
        #
        # GOTCHA: inside a single withColumns() call, every expression is evaluated against the
        # ORIGINAL DataFrame, so col("current_date") does NOT refer to the column created two
        # lines above. It still works because Spark resolves the bare name `current_date` as the
        # built-in current_date function. The clearer, more explicit form is:
        #     current_date() - col("valid_date")
        "delta_date_interval": col("current_date") - col("valid_date")
    })
)

df.display()

## 5. Format dates

**Goal:** convert a `DateType` back into a **string** in a human-friendly layout, for reports, file names, exports or UI display.

`date_format(date, pattern)` uses the same pattern letters as parsing:

| Pattern | Meaning | Example (for 2022-05-18) |
|---------|---------|---------------------------|
| `dd`    | Day, 2 digits | `18` |
| `MM`    | Month number, 2 digits | `05` |
| `MMM`   | Short month name | `May` (e.g. `Jan`, `Feb`, ...) |
| `MMMM`  | Full month name | `May` (e.g. `January`) |
| `yyyy`  | 4-digit year | `2022` |
| `E` / `EEEE` | Day of week (short / full) | `Wed` / `Wednesday` |

So `"dd MMM yyyy"` turns `2022-05-18` into **`18 May 2022`**.

> **Important:** the result is a **string**, so it no longer sorts or computes like a date (`"01 Feb"` sorts before `"18 Jan"`). Do all your date logic on the real date column, and format **only at the end** for presentation.

In [ ]:
# date_format(date_col, pattern) -> StringType
# WHY: dates are stored internally as numbers. This renders them in the exact text layout a reader expects.
from pyspark.sql.functions import date_format

df = (
    # "dd MMM yyyy" -> 2-digit day, short month name, 4-digit year, e.g. "18 May 2022"
    # NULL dates (the invalid ones from step 2) stay NULL.
    df.withColumn("fmt_date", date_format("valid_date", "dd MMM yyyy"))
)

df.display()

## Cheat sheet (quick revision)

```python
from pyspark.sql.functions import (concat_ws, expr, to_date, date_add, date_sub,
                                   add_months, current_date, date_diff, date_format, col)

concat_ws("-", "Y", "M", "D")              # build "2022-5-18" from parts (skips NULLs)
to_date("s", "yyyy-MM-dd")                # string -> date (may THROW on bad input in ANSI mode)
expr("try_to_date(s, 'y-M-d')")           # string -> date, bad input -> NULL (safe)
date_add("d", 5) / date_sub("d", 5)       # +/- days
add_months("d", 5) / add_months("d", -5)  # +/- months (clamps to month end)
current_date()                            # today (same for every row)
date_diff("end", "start")                 # integer days (end first!)
col("end") - col("start")                 # INTERVAL DAY
date_format("d", "dd MMM yyyy")           # date -> display string (do this LAST)
```

### Key takeaways
- **Convert to a real date type early, format to a string late.** Do all logic on `DateType`.
- Spark uses the **Proleptic Gregorian calendar**, so historically "valid" Julian dates such as `1000-02-29` are rejected.
- Choose between **`to_date` (fail fast)** and **`try_to_date` (NULL and continue)** based on how strict the pipeline must be.
- **`M` = month, `m` = minute.** Mixing them up is the most common pattern bug.
- `expr()` is the escape hatch for any **SQL-only** function.
- In one `withColumns()` call, new columns **can't reference each other**. Use a separate `withColumn`/`withColumns` call if one depends on another.